# 01 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 2/27/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

This currently does not use Snowpark File Access as it doesn't yet work in Notebooks. So for now we copy the file locally first.

In [ ]:
import sys
import logging

logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

notebook_name = '01_load_excel_files.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

from snowflake.snowpark.context import get_active_session
try:
    session = get_active_session()
except Exception:
    from snowflake.snowpark.session import _active_sessions
    session = next(iter(_active_sessions))

session.use_schema(f"{database_name}.{schema_name}")

session.use_role(role_name)

current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import re

def read_xlsx_stdlib(file_bytes, sheet_name):
    """Parse an .xlsx file using only stdlib (zipfile + xml). Returns (columns, rows)."""
    with zipfile.ZipFile(BytesIO(file_bytes) if isinstance(file_bytes, bytes) else file_bytes) as zf:
        shared_strings = []
        ss_path = 'xl/sharedStrings.xml'
        if ss_path in zf.namelist():
            ss_tree = ET.parse(zf.open(ss_path))
            ns = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            for si in ss_tree.findall('.//s:si', ns):
                texts = si.findall('.//s:t', ns)
                shared_strings.append(''.join(t.text or '' for t in texts))

        wb_tree = ET.parse(zf.open('xl/workbook.xml'))
        ns_wb = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
                 'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}
        sheet_rid = None
        for s in wb_tree.findall('.//s:sheets/s:sheet', ns_wb):
            if s.get('name') == sheet_name:
                sheet_rid = s.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
                break
        if sheet_rid is None:
            raise ValueError(f"Sheet '{sheet_name}' not found")

        rels_tree = ET.parse(zf.open('xl/_rels/workbook.xml.rels'))
        ns_rel = {'r': 'http://schemas.openxmlformats.org/package/2006/relationships'}
        sheet_file = None
        for rel in rels_tree.findall('.//r:Relationship', ns_rel):
            if rel.get('Id') == sheet_rid:
                sheet_file = 'xl/' + rel.get('Target')
                break

        sheet_tree = ET.parse(zf.open(sheet_file))
        ns_s = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

        def col_letter_to_index(col_str):
            result = 0
            for c in col_str:
                result = result * 26 + (ord(c) - ord('A') + 1)
            return result - 1

        rows_data = []
        for row_el in sheet_tree.findall('.//s:sheetData/s:row', ns_s):
            row_dict = {}
            for cell in row_el.findall('s:c', ns_s):
                ref = cell.get('r')
                col_str = re.match(r'([A-Z]+)', ref).group(1)
                col_idx = col_letter_to_index(col_str)
                cell_type = cell.get('t')
                val_el = cell.find('s:v', ns_s)
                if val_el is not None and val_el.text is not None:
                    if cell_type == 's':
                        val = shared_strings[int(val_el.text)]
                    else:
                        try:
                            val = float(val_el.text)
                            if val == int(val):
                                val = int(val)
                        except ValueError:
                            val = val_el.text
                else:
                    val = None
                row_dict[col_idx] = val
            rows_data.append(row_dict)

        if not rows_data:
            return [], []
        max_col = max(max(r.keys()) for r in rows_data if r) + 1
        header = [rows_data[0].get(i) for i in range(max_col)]
        data_rows = []
        for r in rows_data[1:]:
            data_rows.append([r.get(i) for i in range(max_col)])
        return header, data_rows

print('xlsx stdlib parser ready')

In [ ]:
%%sql -r dataframe_1
-- Temporary solution to load in the metadata, this should be replaced with a directy query to a directory table (or a metadata table)
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
UNION
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION';

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

Note: Until we can use scoped URLs in Notebooks, via the `BUILD_SCOPED_FILE_URL()` function, we need to temporarily copy the file to a temp stage and then process from there.

In [ ]:
from snowflake.snowpark.files import SnowflakeFile
import pandas as pd

session.sql("CREATE TEMP STAGE IF NOT EXISTS temp_excel_stage").collect()

def load_excel_worksheet_to_table(session, external_path, worksheet_name, target_table):
    filename = external_path.split('/')[-1]

    session.sql(f"""
        COPY FILES INTO @temp_excel_stage
        FROM {external_path}
    """).collect()

    with SnowflakeFile.open(f'@temp_excel_stage/{filename}', 'rb') as f:
        file_bytes = f.read()

    columns, rows = read_xlsx_stdlib(file_bytes, worksheet_name)
    df = pd.DataFrame(rows, columns=columns)

    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("overwrite").save_as_table(target_table)

    logger.info(f"Loaded {len(df)} rows from '{worksheet_name}' to {target_table}", extra = {'logger_name': logger_name})

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
# Process each file from the sql_get_spreadsheets cell above
files_to_load = dataframe_1
for index, excel_file in files_to_load.iterrows():
    print(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info(f"Finish executing notebook {notebook_name}", extra = {'logger_name': logger_name})

### Debugging

In [ ]:
%%sql -r dataframe_2
--DESCRIBE TABLE LOCATION;
--SELECT * FROM LOCATION;
SHOW TABLES;